In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# In VSCode, right click on train.csv --> copy path, paste here
path = "/Users/matteogiardina/Desktop/BEMACS 2/2nd semester/ML Project/Data/train.csv"
df = pd.read_csv(path)
df = df.drop(columns=["Unnamed: 0"])
print(len(df))

110930


In [3]:
# The exact format from your screenshot: MM/DD/YYYY HH:MM:SS AM/PM
date_format = '%m/%d/%Y %I:%M:%S %p'

# Convert strings to actual pandas datetime objects
df['Created Date'] = pd.to_datetime(df['Created Date'], format=date_format, errors='coerce')
df['Closed Date'] = pd.to_datetime(df['Closed Date'], format=date_format, errors='coerce')

# Drop any rows where the Created Date was corrupted or missing
df = df.dropna(subset=['Created Date'])

# Find the absolute latest date in the entire dataset (Data Dump Date)
max_date = max(df['Created Date'].max(), df['Closed Date'].max())
print(f"   -> Dataset extracted roughly around: {max_date}")

# Calculate how old every ticket was at the exact moment the data was pulled
df['Age at Data Dump'] = max_date - df['Created Date']

# Identify open tickets that are less than 24 hours old
unknown_outcomes = df['Closed Date'].isna() & (df['Age at Data Dump'] <= pd.Timedelta(days=1))
print(f"   -> Dropping {unknown_outcomes.sum()} rows because their 24-hour window hasn't expired yet.")

# Filter out those unknown rows
df = df[~unknown_outcomes].copy()

# Calculate time to close
df['Time to Close'] = df['Closed Date'] - df['Created Date']

# Create Y: 1 if closed within 24 hours, else 0
df['Y'] = (df['Time to Close'] <= pd.Timedelta(days=1)).astype(int)

print("\n--- DONE ---")
print("Baseline Accuracy (Distribution of Y):")
print(df['Y'].value_counts(normalize=True) * 100)

   -> Dataset extracted roughly around: 2026-04-19 01:35:00
   -> Dropping 0 rows because their 24-hour window hasn't expired yet.

--- DONE ---
Baseline Accuracy (Distribution of Y):
Y
1    61.387361
0    38.612639
Name: proportion, dtype: float64


In [4]:
# Extracting numerical time features from the Created Date
df['Created_Hour'] = df['Created Date'].dt.hour           # Returns 0-23
df['Created_DayOfWeek'] = df['Created Date'].dt.dayofweek # Returns 0-6 (0=Monday, 6=Sunday)
df['Created_Month'] = df['Created Date'].dt.month         # Returns 1-12

# You can even create custom binary features!
# 1 if it's Saturday (5) or Sunday (6), else 0
df['Is_Weekend'] = df['Created_DayOfWeek'].isin([5, 6]).astype(int) 

In [5]:
df.head()

,Unique Key,Created Date,Closed Date,Agency,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Additional Details,Location Type,Incident Zip,...,Latitude,Longitude,Location,Age at Data Dump,Time to Close,Y,Created_Hour,Created_DayOfWeek,Created_Month,Is_Weekend
0,68670099,2026-04-14 08:21:30,2026-04-15 02:20:44,HPD,Department of Housing Preservation and Develop...,HEAT/HOT WATER,ENTIRE BUILDING,NO HEAT AND NO HOT WATER,RESIDENTIAL BUILDING,10011.0,...,"40,746983691","-74,00646371084",POINT (-74.006463710836 40.746983690998),4 days 17:13:30,0 days 17:59:14,1,8,1,4,0
1,68613406,2026-04-09 15:54:00,2026-04-11 02:18:59,HPD,Department of Housing Preservation and Develop...,HEAT/HOT WATER,APARTMENT ONLY,NO HEAT AND NO HOT WATER,RESIDENTIAL BUILDING,10025.0,...,"40,80108226257","-73,96317277601",POINT (-73.963172776011 40.801082262571),9 days 09:41:00,1 days 10:24:59,0,15,3,4,0
2,68569602,2026-04-05 12:18:59,2026-04-08 15:05:09,DSNY,Department of Sanitation,Illegal Dumping,Removal Request,NaN,Sidewalk,11238.0,...,"40,68620386229","-73,96432875102",POINT (-73.964328751024 40.68620386229),13 days 13:16:01,3 days 02:46:10,0,12,6,4,1
3,68556726,2026-04-03 12:09:10,2026-04-06 00:00:00,DOB,Department of Buildings,Real Time Enforcement,Work Without A Permit - Occupied Multiple Dwel...,NaN,NaN,10453.0,...,"40,85129107617","-73,91196506395",POINT (-73.911965063954 40.851291076167),15 days 13:25:50,2 days 11:50:50,0,12,4,4,0
4,68540433,2026-04-02 09:18:56,NaT,HPD,Department of Housing Preservation and Develop...,ELECTRIC,OUTLET/SWITCH,OUTLET SEALED OR DEFECTIVE,RESIDENTIAL BUILDING,10016.0,...,"40,74152278859","-73,97977323839",POINT (-73.979773238387 40.741522788589),16 days 16:16:04,NaT,0,9,3,4,0
